In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\taral\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("Dataset/reviews.csv")
df["review_text"].head(20)

0                                             It's good
1     WhatsApp not working well always shows offline...
2     Oppo not corresponding, share with me the offi...
3     Excellent app, great communication super conne...
4          simply the ɓest for chat and calls.i love it
5            good. but i need WhatsApp premium features
6     learning learning learning learning learning l...
7       Awesome. I just need it to download and install
8                            very nice app thnx so much
9                          Really really apriacite 100/
10    whatsapp web not working camera Focus not working
11                                 Nice and good to use
12    This account can no longer use watsapp due to ...
13    forever controversy workshop lawyer carbon ins...
14    I've been using this app for years and I just ...
15    AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...
16    They banned my account more than 9 times I did...
17    i can't open my whatsapp whenever i open i

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6210 entries, 0 to 6209
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    6210 non-null   object
 1   rating       6210 non-null   int64 
 2   review_text  6210 non-null   object
 3   review_date  6210 non-null   object
 4   helpful      6210 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 242.7+ KB


In [4]:
#Text preprocessing
stop_words = set(stopwords.words("english"))

words_to_keep = {
    "not", "no", "nor", "never", "neither", "nothing", "very", "too", "most", "least",
    "more", "less", "least", "few", "but", "however", "although", "just", "only", "above", "below"
}

stop_words = stop_words - words_to_keep

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    words= ([word for word in text.split() if word not in stop_words])
    
    return " ".join(words)


df["clean_text"] = df["review_text"].apply(clean_text)

df = df[df["clean_text"].str.len() > 0].reset_index(drop = True)

print("\nRemaning rows:", df.shape)


Remaning rows: (6168, 6)


In [5]:
df["clean_text"].head(20)

0                                                  good
1     whatsapp not working well always shows offline...
2       oppo not corresponding share official app oppor
3     excellent app great communication super connec...
4                           simply est chat callsi love
5               good but need whatsapp premium features
6     learning learning learning learning learning l...
7                    awesome just need download install
8                               very nice app thnx much
9                               really really apriacite
10    whatsapp web not working camera focus not working
11                                        nice good use
12    account no longer use watsapp due spam kyu ho ...
13    forever controversy workshop lawyer carbon ins...
14    ive using app years just absolutely love featu...
15    aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa...
16                 banned account more times didnt know
17    cant open whatsapp whenever open close aut

In [ ]:
#Convert text to numeric
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,3),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

In [7]:
#Split the data for training
x = vectorizer.fit_transform(df["clean_text"])
y = df["rating"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, 
    test_size = 0.2,
    random_state = 42) 

In [ ]:
#Setting the parameters for the model and training it
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators = 300,
    learning_rate = 0.01,
    num_leaves = 31,
    max_depth = -1
)

model.fit(x_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007266 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8234
[LightGBM] [Info] Number of data points in the train set: 4934, number of used features: 380
[LightGBM] [Info] Start training from score -1.369812
[LightGBM] [Info] Start training from score -2.609502
[LightGBM] [Info] Start training from score -2.532643
[LightGBM] [Info] Start training from score -2.279347
[LightGBM] [Info] Start training from score -0.712382


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.01
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [9]:
y_prediction = model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_prediction))
print(classification_report(y_test, y_prediction))

Accuracy: 0.5980551053484603
              precision    recall  f1-score   support

           1       0.57      0.61      0.59       319
           2       0.18      0.02      0.04        92
           3       0.00      0.00      0.00       110
           4       0.07      0.01      0.02       117
           5       0.63      0.90      0.74       596

    accuracy                           0.60      1234
   macro avg       0.29      0.31      0.28      1234
weighted avg       0.47      0.60      0.52      1234



c:\Anaconda\envs\m1_exam\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
feature_names = vectorizer.get_feature_names_out()

importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values(by = "importance", ascending = False)

print(importance.head(20))

       feature  importance
4815  whatsapp        1559
219        app        1125
2885       not        1092
1862      good         954
3247    please         877
4624      very         808
2831      nice         731
735       cant         711
2516      love         700
656        but         671
1930     great         610
31     account         609
527       best         577
2426      like         527
4544       use         522
1328  download         407
3400   problem         405
3060      open         387
2014      help         364
3511    really         360


In [ ]:
#Grid search to find the best parameters for the model
from sklearn.model_selection import GridSearchCV
model = lgb.LGBMClassifier()

param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.03, 0.1],
    "num_leaves": [31, 50, 70],
    "max_depth": [-1, 10, 20]
}

grid = GridSearchCV(
    estimator = model,
    param_grid = param_grid,
    cv = 3,
    scoring = "accuracy",
    verbose = 1,
    n_jobs = -1
)

grid.fit(x_train, y_train)
print("Best parameters:", grid.best_params_)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005243 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8190
[LightGBM] [Info] Number of data points in the train set: 4934, number of used features: 375
[LightGBM] [Info] Start training from score -1.369812
[LightGBM] [Info] Start training from score -2.609502
[LightGBM] [Info] Start training from score -2.532643
[LightGBM] [Info] Start training from score -2.279347
[LightGBM] [Info] Start training from score -0.712382
Best parameters: {'learning_rate': 0.01, 'max_depth': -1, 'n_estimators': 300, 'num_leaves': 31}
